In [1]:
%pip install openai scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1", # Ollama
    api_key="sk-1234"  # Dummy key
)

In [3]:
# Set model parameters

SYSTEM_PROMPT = """
You are for a GAD-7 survey chatbot.
Your job is to extract answers for GAD-7 based on a given transcript.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a widely used self-administered diagnostic tool designed to screen for and assess the severity of generalized anxiety disorder (GAD).
GAD-7 question order:
q1: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
q2: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
q3: Over the last two weeks, how often have you been bothered by worrying too much about different things?
q4: Over the last two weeks, how often have you been bothered by trouble relaxing?
q5: Over the last two weeks, how often have you been bothered by being so restless that it is hard to sit still?
q6: Over the last two weeks, how often have you been bothered by becoming easily annoyed or irritable?
q7: Over the last two weeks, how often have you been bothered by feeling afraid, as if something awful might happen?
Scale mapping:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day


Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- For each question q1-q7, determine the score 0-3 based on the user's responses.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q7 key.
- expected_output must contain exactly q1 through q7.
- conversation must be consistent with expected_output.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
"""
MODEL = 'gpt-oss-20b@mxfp4'

- Use a json dataset of conversation transcript
- Extract scores using LLM or NLP technique
- Validate whether they were correct

In [4]:
# Load the GAD dataset
import json 

with open('gad_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')

SUCCESS: Number of conversations matches number of scores


In [5]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(len(conv_train), len(_conv_test), len(score_train), len(_score_test))

7 3 7 3


In [6]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript: str) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(transcript)
        ],
    )

    return completion.choices[0].message.content

In [7]:
scores = []
for i, c in enumerate(conv_train):
    print(f'Generating {i+1}/{len(conv_train)}')
    scores.append(create_prediction(json.dumps(c)))
    print(scores[-1])

Generating 1/7
{
  "q1": "1",
  "q2": "1",
  "q3": "1",
  "q4": "1",
  "q5": "1",
  "q6": "2",
  "q7": "0"
}
Generating 2/7
{
  "q1": "1",
  "q2": "2",
  "q3": "1",
  "q4": "2",
  "q5": "3",
  "q6": "1",
  "q7": "1"
}
Generating 3/7
{"q1":"1","q2":"2","q3":"1","q4":"2","q5":"1","q6":"2","q7":"1"}
Generating 4/7
{"q1":"1","q2":"2","q3":"2","q4":"1","q5":"1","q6":"2","q7":"1"}
Generating 5/7
{
  "q1": "1",
  "q2": "1",
  "q3": "1",
  "q4": "1",
  "q5": "1",
  "q6": "1",
  "q7": "1"
}
Generating 6/7
{"q1":"1","q2":"1","q3":"2","q4":"2","q5":"1","q6":"1","q7":"1"}
Generating 7/7
{"q1":"1","q2":"0","q3":"1","q4":"2","q5":"1","q6":"1","q7":"1"}


In [8]:
json_scores = [json.loads(s) for s in scores]

In [9]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores):
    return [int(s) if s else 0 for s in scores.values.flatten().tolist()]

pred_df = pd.DataFrame(json_scores)
train_df = pd.DataFrame(score_train)

mse = mean_squared_error(
    convert_scores_to_array(train_df),
    convert_scores_to_array(pred_df)
)
print(mse) 

diff_df = train_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(columns={'self': 'Expected', 'other': 'Predicted'}, level=1)
display(diff_df)

0.0


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        1         1        1         1        1         1        1         1   
1        1         1        2         2        1         1        2         2   
2        1         1        2         2        1         1        2         2   
3        1         1        2         2        2         2        1         1   
4        1         1        1         1        1         1        1         1   
5        1         1        1         1        2         2        2         2   
6        1         1        0         0        1         1        2         2   

        q5                 q6                 q7            
  Expected Predicted Expected Predicted Expected Predicted  
0        1         1        2         2        0         0  
1        3         3        1         1        1         1  
2        1         1        2         2        1         1  
3        1         1        2         2        1         1  
4        1         1        1         1        1         1  
5        1         1        1         1        1         1  
6        1         1        1         1        1         1